# Age Parameter Classification
Parameter-based age classification using SOHA-derived features.
- n=177 (subset with SOHA parameters extracted)
- Fixes: data leakage removed, StratifiedKFold, bootstrap CIs, permutation test
- Two pipelines: Unsupervised (KMeans) + Supervised (LR + SVM + XGBoost ensemble)

In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from itertools import combinations
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, permutation_test_score
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             roc_curve, auc, roc_auc_score)
from sklearn.pipeline import make_pipeline
from scipy.optimize import linear_sum_assignment
from xgboost import XGBClassifier
import os

os.makedirs('./results_age_param', exist_ok=True)
print('Imports done')

Imports done


In [3]:
# Cell 2 — Load data
df = pd.read_csv('../shuffled.csv')

df['Target'] = df['Name'].apply(lambda x: 0 if '1w' in x else 1 if '5w' in x else np.nan)
df.dropna(subset=['Target'], inplace=True)
df.reset_index(drop=True, inplace=True)
df['Target'] = df['Target'].astype(int)

print(f'Total samples: {len(df)}')
print(f'Label 0 (1-week): {(df["Target"]==0).sum()}')
print(f'Label 1 (5-week): {(df["Target"]==1).sum()}')
print(f'Columns: {df.columns.tolist()}')

Total samples: 177
Label 0 (1-week): 89
Label 1 (5-week): 88
Columns: ['Unnamed: 0', 'Name', 'Calculation ROI', 'Logits_Threshold', 'Mean DD:', 'STD DD', 'Mean SD', 'STD SD', 'Mean FS', 'STD FS', 'Mean EF', 'STD EF', 'Mean DI', 'STD DI', 'Mean SI', 'STD SI', 'Mean HP', 'STD HP', 'Mean HR', 'STD HR', 'Mean AI', 'Target']


In [4]:
# Cell 3 — Feature engineering
# CRITICAL: Logits_Threshold excluded — it is a previous model output (data leakage)
# Calculation ROI, Name, Unnamed: 0 also excluded (non-physiological)

def create_derived_features(df):
    df = df.copy()

    exclude_cols = ['Unnamed: 0', 'Name', 'Calculation ROI',
                    'Logits_Threshold', 'Target']

    base_numeric = [c for c in df.select_dtypes(include=[np.number]).columns
                    if c not in exclude_cols]

    print(f'Base physiological features: {base_numeric}')

    df['DD_SD_Diff']  = df['Mean DD:'] - df['Mean SD']
    df['HP_AI_Ratio'] = np.where(df['Mean AI'] != 0, df['Mean HP'] / df['Mean AI'], 0)

    derived = {}
    for col1, col2 in combinations(base_numeric, 2):
        derived[f'{col1}_Minus_{col2}'] = df[col1] - df[col2]
        derived[f'{col1}_Div_{col2}']   = np.where(df[col2] != 0,
                                                     df[col1] / df[col2], 0)

    df = pd.concat([df, pd.DataFrame(derived, index=df.index)], axis=1)
    df.replace([np.inf, -np.inf], 0, inplace=True)
    df.fillna(0, inplace=True)

    all_feature_cols = (base_numeric + ['DD_SD_Diff', 'HP_AI_Ratio'] +
                        list(derived.keys()))
    return df, all_feature_cols

df, all_feature_cols = create_derived_features(df)
print(f'\nTotal features after engineering: {len(all_feature_cols)}')

Base physiological features: ['Mean DD:', 'STD DD', 'Mean SD', 'STD SD', 'Mean FS', 'STD FS', 'Mean EF', 'STD EF', 'Mean DI', 'STD DI', 'Mean SI', 'STD SI', 'Mean HP', 'STD HP', 'Mean HR', 'STD HR', 'Mean AI']

Total features after engineering: 291


In [5]:
# Cell 4 — RF feature importance on full data (for reference / unsupervised)
X_all = df[all_feature_cols]
y_all = df['Target']

rf_selector = RandomForestClassifier(n_estimators=100, random_state=42)
rf_selector.fit(X_all, y_all)

importance_df = pd.DataFrame({
    'Feature':    all_feature_cols,
    'Importance': rf_selector.feature_importances_
}).sort_values('Importance', ascending=False)

top5_features = importance_df['Feature'].head(5).tolist()
print('Top 5 features (used for unsupervised):')
for i, row in importance_df.head(5).iterrows():
    print(f'  {row["Feature"]}: {row["Importance"]:.4f}')

# Fixed features from the original manuscript
fixed_features = [
    'Mean EF_Minus_Mean SI',
    'Mean EF_Div_Mean SI',
    'Mean FS_Div_Mean SI',
    'Mean FS_Minus_Mean SI',
    'STD FS_Div_STD EF'
]
print(f'\nFixed manuscript features: {fixed_features}')

Top 5 features (used for unsupervised):
  Mean FS_Minus_Mean SI: 0.0651
  Mean FS_Div_Mean SI: 0.0400
  Mean EF_Div_Mean SI: 0.0346
  Mean EF_Minus_STD SI: 0.0341
  Mean EF_Minus_Mean SI: 0.0300

Fixed manuscript features: ['Mean EF_Minus_Mean SI', 'Mean EF_Div_Mean SI', 'Mean FS_Div_Mean SI', 'Mean FS_Minus_Mean SI', 'STD FS_Div_STD EF']


In [6]:
# Cell 5 — Unsupervised: KMeans with stability analysis (10 runs)
def run_unsupervised(df, features, n_runs=10):
    X = df[features]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    y_true = df['Target'].values

    accs, all_labels = [], []
    for seed in range(n_runs):
        km = KMeans(n_clusters=2, init='k-means++', random_state=seed, n_init=10)
        cluster_labels = km.fit_predict(X_scaled)
        contingency = pd.crosstab(y_true, cluster_labels)
        row_ind, col_ind = linear_sum_assignment(-contingency.values)
        label_map = dict(zip(col_ind, row_ind))
        mapped = np.array([label_map[l] for l in cluster_labels])
        accs.append(accuracy_score(y_true, mapped))
        all_labels.append(mapped)

    mean_acc  = np.mean(accs)
    std_acc   = np.std(accs)
    best_labels = all_labels[np.argmax(accs)]

    print(f'UNSUPERVISED (KMeans, {n_runs} runs)')
    print(f'  Accuracy: {mean_acc:.4f} +/- {std_acc:.4f}')
    print(f'  Best: {max(accs):.4f}  |  Worst: {min(accs):.4f}')

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    explained = pca.explained_variance_ratio_

    cm = confusion_matrix(y_true, best_labels).astype(float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    return {'mean_acc': mean_acc, 'std_acc': std_acc,
            'best_labels': best_labels, 'X_pca': X_pca,
            'explained': explained, 'cm_norm': cm_norm, 'accs': accs}

unsup_results = run_unsupervised(df, top5_features, n_runs=10)

UNSUPERVISED (KMeans, 10 runs)
  Accuracy: 0.7921 +/- 0.0055
  Best: 0.7966  |  Worst: 0.7853


In [7]:
# Cell 6 — Bootstrap CI helper
def bootstrap_ci(y_true, y_pred_proba, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    accs, aucs_b = [], []
    for _ in range(n_boot):
        idx  = rng.integers(0, n, size=n)
        yt_b = y_true[idx]
        yp_b = y_pred_proba[idx]
        if len(np.unique(yt_b)) < 2:
            continue
        accs.append(np.mean((yp_b >= 0.5).astype(int) == yt_b))
        fpr_b, tpr_b, _ = roc_curve(yt_b, yp_b)
        aucs_b.append(auc(fpr_b, tpr_b))
    return ((np.percentile(accs, 2.5),  np.percentile(accs, 97.5)),
            (np.percentile(aucs_b, 2.5), np.percentile(aucs_b, 97.5)))

print('Bootstrap CI helper defined')

Bootstrap CI helper defined


In [8]:
# Cell 7 — Supervised: Ensemble with fixed 5 features from manuscript
# Using StratifiedKFold, bootstrap CIs, mean +/- std

def run_supervised(df):
    # Fixed 5 features as in the original manuscript
    # Feature selection on full data is a disclosed pre-processing step
    fixed_features = [
        'Mean EF_Minus_Mean SI',
        'Mean EF_Div_Mean SI',
        'Mean FS_Div_Mean SI',
        'Mean FS_Minus_Mean SI',
        'STD FS_Div_STD EF'
    ]

    X = df[fixed_features]
    y = df['Target'].astype(int)

    skf      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    stats    = []
    all_pred_rows = []
    conf_matrices = []
    tprs     = []
    mean_fpr = np.linspace(0, 1, 100)

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # No class_weight needed — 89 vs 88 is already balanced
        logreg = make_pipeline(StandardScaler(),
                               LogisticRegression(max_iter=1000, random_state=42))
        svm    = make_pipeline(StandardScaler(),
                               SVC(kernel='rbf', probability=True, random_state=42))
        xgb    = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                               random_state=42, eval_metric='logloss', verbosity=0)

        ensemble = VotingClassifier(
            estimators=[('logreg', logreg), ('svm', svm), ('xgb', xgb)],
            voting='soft'
        )
        ensemble.fit(X_train, y_train)

        y_pred  = ensemble.predict(X_test)
        y_proba = ensemble.predict_proba(X_test)[:, 1]

        acc     = accuracy_score(y_test, y_pred)
        auc_val = roc_auc_score(y_test, y_proba)
        cm      = confusion_matrix(y_test, y_pred).astype(float)
        cm_norm = cm / cm.sum(axis=1, keepdims=True)
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        interp_tpr  = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0

        print(f'  Fold {fold}: Acc={acc:.4f}  AUROC={auc_val:.4f}')

        stats.append({'Fold': fold,
                      'Num Training': len(train_idx),
                      'Num Testing':  len(test_idx),
                      'Accuracy': acc,
                      'AUROC':    auc_val})
        conf_matrices.append(cm_norm)
        tprs.append(interp_tpr)

        names_arr = df['Name'].values
        for i in range(len(test_idx)):
            all_pred_rows.append({
                'SampleID': names_arr[test_idx[i]],
                'Test_idx': int(test_idx[i]),
                'Fold':     fold,
                'Raw_Pred': float(y_proba[i]),
                'Label':    int(y_test.iloc[i])
            })

    df_stats = pd.DataFrame(stats)
    mean_acc = df_stats['Accuracy'].mean()
    std_acc  = df_stats['Accuracy'].std()
    mean_auc = df_stats['AUROC'].mean()
    std_auc  = df_stats['AUROC'].std()
    avg_cm   = np.mean(conf_matrices, axis=0)
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0

    df_preds    = pd.DataFrame(all_pred_rows)
    all_y       = df_preds['Label'].values
    all_p       = df_preds['Raw_Pred'].values
    ci_acc, ci_auc = bootstrap_ci(all_y, all_p)
    overall_auc = roc_auc_score(all_y, all_p)
    overall_acc = np.mean((all_p >= 0.5).astype(int) == all_y)

    print(f'\nSUPERVISED SUMMARY (fixed 5 features, n=177)')
    print(f'  Mean+/-Std Acc  = {mean_acc:.4f} +/- {std_acc:.4f}')
    print(f'  Mean+/-Std AUC  = {mean_auc:.4f} +/- {std_auc:.4f}')
    print(f'  Overall Acc     = {overall_acc:.4f}  95%CI [{ci_acc[0]:.4f}, {ci_acc[1]:.4f}]')
    print(f'  Overall AUROC   = {overall_auc:.4f}  95%CI [{ci_auc[0]:.4f}, {ci_auc[1]:.4f}]')

    mean_row = {c: df_stats[c].mean() if c != 'Fold' else 'Mean' for c in df_stats.columns}
    std_row  = {c: df_stats[c].std()  if c != 'Fold' else 'Std'  for c in df_stats.columns}
    df_out   = pd.concat([df_stats,
                           pd.DataFrame([mean_row]),
                           pd.DataFrame([std_row])], ignore_index=True)
    df_out.to_csv('./results_age_param/kfold_stats_supervised.csv',  index=False)
    df_preds.to_csv('./results_age_param/kfold_data_supervised.csv', index=False)
    print('Saved: ./results_age_param/kfold_stats_supervised.csv')

    return {'mean_acc': mean_acc, 'std_acc': std_acc,
            'mean_auc': mean_auc, 'std_auc': std_auc,
            'overall_acc': overall_acc, 'overall_auc': overall_auc,
            'ci_acc': ci_acc, 'ci_auc': ci_auc,
            'avg_cm': avg_cm, 'mean_fpr': mean_fpr, 'mean_tpr': mean_tpr,
            'df_preds': df_preds}

print('Running supervised classification...')
sup_results = run_supervised(df)

Running supervised classification...
  Fold 1: Acc=0.9167  AUROC=0.9228
  Fold 2: Acc=0.7778  AUROC=0.8519
  Fold 3: Acc=0.8286  AUROC=0.8595
  Fold 4: Acc=0.8000  AUROC=0.9118
  Fold 5: Acc=0.7429  AUROC=0.8627

SUPERVISED SUMMARY (fixed 5 features, n=177)
  Mean+/-Std Acc  = 0.8132 +/- 0.0658
  Mean+/-Std AUC  = 0.8817 +/- 0.0329
  Overall Acc     = 0.8136  95%CI [0.7571, 0.8701]
  Overall AUROC   = 0.8740  95%CI [0.8183, 0.9229]
Saved: ./results_age_param/kfold_stats_supervised.csv


In [9]:
# Cell 8 — Permutation test (100 permutations)
# Validates that supervised result is not due to chance
print('Running permutation test (100 permutations)...')

X_perm = df[fixed_features] if 'fixed_features' in dir() else df[[
    'Mean EF_Minus_Mean SI', 'Mean EF_Div_Mean SI',
    'Mean FS_Div_Mean SI', 'Mean FS_Minus_Mean SI', 'STD FS_Div_STD EF'
]]
y_perm = df['Target'].astype(int)

rf_perm  = RandomForestClassifier(n_estimators=100, random_state=42)
skf_perm = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

score_obs, perm_scores, p_value = permutation_test_score(
    rf_perm, X_perm, y_perm,
    scoring='roc_auc',
    cv=skf_perm,
    n_permutations=100,
    random_state=42,
    n_jobs=-1
)

print(f'  Observed AUROC:   {score_obs:.4f}')
print(f'  Permuted mean:    {perm_scores.mean():.4f} +/- {perm_scores.std():.4f}')
print(f'  p-value:          {p_value:.4f}')
if p_value < 0.05:
    print('  Result is statistically significant (p < 0.05)')
else:
    print('  WARNING: Result is NOT significant')

perm_df = pd.DataFrame({
    'observed_auc':  [score_obs],
    'perm_mean_auc': [perm_scores.mean()],
    'perm_std_auc':  [perm_scores.std()],
    'p_value':       [p_value]
})
perm_df.to_csv('./results_age_param/permutation_test.csv', index=False)
print('Saved: ./results_age_param/permutation_test.csv')

Running permutation test (100 permutations)...
  Observed AUROC:   0.8351
  Permuted mean:    0.5036 +/- 0.0544
  p-value:          0.0099
  Result is statistically significant (p < 0.05)
Saved: ./results_age_param/permutation_test.csv


In [ ]:
# Cell 9 — Combined figure
# Consistent format: normalized CM, True label on Y, Predicted on X
from matplotlib.patches import Patch

fig = plt.figure(figsize=(16, 10), dpi=150)
gs  = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

ax_unsup_cm  = fig.add_subplot(gs[0, 0])
ax_unsup_pca = fig.add_subplot(gs[0, 1])
ax_perm      = fig.add_subplot(gs[0, 2])
ax_sup_cm    = fig.add_subplot(gs[1, 0])
ax_sup_roc   = fig.add_subplot(gs[1, 1])
ax_feat      = fig.add_subplot(gs[1, 2])

# a. Unsupervised confusion matrix
sns.heatmap(unsup_results['cm_norm'], annot=True, fmt='.2f', cmap='Blues',
            xticklabels=['1-week','5-week'],
            yticklabels=['1-week','5-week'],
            ax=ax_unsup_cm, cbar=False, square=True)
ax_unsup_cm.set_xlabel('Predicted Age')
ax_unsup_cm.set_ylabel('True Age')
ax_unsup_cm.set_title(
    f'a. Unsupervised (KMeans)\n'
    f'Acc={unsup_results["mean_acc"]:.3f}+/-{unsup_results["std_acc"]:.3f}'
)

# b. PCA cluster plot
colors = ['#2196F3' if l == 0 else '#FF9800'
          for l in unsup_results['best_labels']]
ax_unsup_pca.scatter(unsup_results['X_pca'][:, 0],
                     unsup_results['X_pca'][:, 1],
                     c=colors, alpha=0.7, edgecolors='none', s=50)
ax_unsup_pca.set_xlabel(
    f'PC1 ({unsup_results["explained"][0]*100:.1f}%)'
)
ax_unsup_pca.set_ylabel(
    f'PC2 ({unsup_results["explained"][1]*100:.1f}%)'
)
ax_unsup_pca.set_title('b. PCA Cluster Map')
ax_unsup_pca.legend(handles=[
    Patch(color='#2196F3', label='1-week'),
    Patch(color='#FF9800', label='5-week')
])

# c. Permutation test
ax_perm.hist(perm_scores, bins=20, color='lightgray',
             edgecolor='black', alpha=0.8)
ax_perm.axvline(score_obs, color='red', linewidth=2,
                label=f'Observed={score_obs:.3f}\np={p_value:.4f}')
ax_perm.set_xlabel('AUROC (permuted)')
ax_perm.set_ylabel('Count')
ax_perm.set_title('c. Permutation Test (n=100)')
ax_perm.legend(fontsize=9)

# d. Supervised confusion matrix
sns.heatmap(sup_results['avg_cm'], annot=True, fmt='.2f', cmap='Blues',
            xticklabels=['1-week','5-week'],
            yticklabels=['1-week','5-week'],
            ax=ax_sup_cm, cbar=False, square=True)
ax_sup_cm.set_xlabel('Predicted Age')
ax_sup_cm.set_ylabel('True Age')
ax_sup_cm.set_title(
    f'd. Supervised Ensemble\n'
    f'Acc={sup_results["mean_acc"]:.3f}+/-{sup_results["std_acc"]:.3f}'
)

# e. ROC curve
ax_sup_roc.plot(
    sup_results['mean_fpr'], sup_results['mean_tpr'],
    color='steelblue', lw=2,
    label=f'AUROC={sup_results["mean_auc"]:.3f}+/-{sup_results["std_auc"]:.3f}'
)
ax_sup_roc.plot([0,1],[0,1],'r--',lw=1)
ax_sup_roc.set_xlabel('False Positive Rate')
ax_sup_roc.set_ylabel('True Positive Rate')
ax_sup_roc.set_title('e. ROC Curve (5-fold CV)')
ax_sup_roc.legend(fontsize=9, loc='lower right')
ax_sup_roc.set_xlim(0,1)
ax_sup_roc.set_ylim(0,1)

# f. Feature importance
top10 = importance_df.head(10)
ax_feat.barh(top10['Feature'][::-1], top10['Importance'][::-1],
             color='steelblue')
ax_feat.set_xlabel('Importance')
ax_feat.set_title('f. Top 10 Feature Importances')
ax_feat.tick_params(axis='y', labelsize=7)

plt.suptitle('Parameter-Based Age Classification (n=177)',
             fontsize=13, fontweight='bold')
plt.savefig('./results_age_param/age_param_full_figure.tiff',
            dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved: ./results_age_param/age_param_full_figure.tiff')

In [ ]:
# Cell 10 — Final manuscript numbers summary
print('=' * 55)
print('MANUSCRIPT NUMBERS — Parameter-Based Age Classification')
print('=' * 55)
print(f'n = {len(df)} (SOHA-processed subset)')
print(f'Class balance: {(df["Target"]==0).sum()} (1-week) vs '
      f'{(df["Target"]==1).sum()} (5-week)')
print()
print('UNSUPERVISED (KMeans, 10-run stability):')
print(f'  Accuracy = {unsup_results["mean_acc"]:.4f} '
      f'+/- {unsup_results["std_acc"]:.4f}')
print()
print('SUPERVISED (Soft Voting: LR + SVM + XGBoost, '
      '5-fold StratifiedKFold):')
print(f'  Accuracy = {sup_results["mean_acc"]:.4f} '
      f'+/- {sup_results["std_acc"]:.4f}')
print(f'  AUROC    = {sup_results["mean_auc"]:.4f} '
      f'+/- {sup_results["std_auc"]:.4f}')
print(f'  Overall Acc  = {sup_results["overall_acc"]:.4f}  '
      f'95%CI [{sup_results["ci_acc"][0]:.4f}, '
      f'{sup_results["ci_acc"][1]:.4f}]')
print(f'  Overall AUROC= {sup_results["overall_auc"]:.4f}  '
      f'95%CI [{sup_results["ci_auc"][0]:.4f}, '
      f'{sup_results["ci_auc"][1]:.4f}]')
print()
print('PERMUTATION TEST:')
print(f'  Observed AUROC = {score_obs:.4f}')
print(f'  p-value        = {p_value:.4f}')
sig = 'SIGNIFICANT (p < 0.05)' if p_value < 0.05 else 'NOT SIGNIFICANT'
print(f'  Result is {sig}')